# Lab 2.1 — Open-Source LLMs: First Contact
**Module II · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PLACEHOLDER/labs_solutions/blob/main/module-2-llm/lab2_1_open_source_llms.ipynb)

---

## What you will do
1. Load a local open-source LLM through a unified interface that works both locally (Ollama) and on Colab (HuggingFace fallback).
2. Generate text and observe how **temperature** controls creativity vs. consistency.
3. Deliberately trigger a **hallucination** by asking the LLM about information it cannot know — and understand *why* this happens structurally.
4. `[Extension]` Understand **context windows** and token counting.

## Prerequisites
Module I completed (or equivalent Python comfort). No prior LLM experience needed.

**Estimated time:** 40–50 min

---
## 0 · Setup

If you are running **locally with Ollama**: make sure Ollama is running (`ollama serve`) and that you have pulled a model:
```bash
ollama pull llama3.2:1b
```

If you are on **Google Colab** or Ollama is not running: the LLM wrapper will automatically download a small (~3 GB) HuggingFace model. This takes a few minutes the first time but is then cached.

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/PLACEHOLDER/labs_solutions.git"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         "labs_solutions/environment/requirements.txt"], check=True)
    sys.path.insert(0, "labs_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
from utils.llm import SimpleLLM
from utils import load_company_kb
print("Imports OK.")

---
## 1 · The LLM wrapper

We use a thin `SimpleLLM` class that hides whether you are talking to Ollama or a HuggingFace model. The API is the same either way:

```python
llm.generate(prompt)              # single user message → reply
llm.chat(messages)                # list of {role, content} dicts → reply
```

The cell below creates the wrapper and auto-detects the backend.

In [ ]:
llm = SimpleLLM()
print(llm)

---
## 2 · Basic generation

### Exercise 2.1.1 `[Core]` — Hello, LLM

Ask the LLM to explain what machine learning is, in one sentence, as if speaking to a business executive with no technical background. Use `llm.generate(prompt)` and print the response.

In [ ]:
# --- SOLUTION ---
prompt = (
    "Explain what machine learning is in one sentence, "
    "for a business executive with no technical background."
)
response = llm.generate(prompt)
print(response)

> **Notice:** The model produces a fluent, contextually appropriate answer — without being explicitly programmed with facts about machine learning. That is the power of large-scale pre-training.

---
## 3 · Temperature: creativity vs. consistency

LLM outputs are probabilistic — the model samples from a probability distribution over possible next tokens. **Temperature** controls how sharp or flat that distribution is:
- `temperature = 0` → always picks the most probable token → **deterministic, repetitive**.
- `temperature = 1` → samples proportionally to probabilities → **diverse, sometimes surprising**.
- `temperature > 1` → flattens distribution → **very creative but also unreliable**.

### Exercise 2.1.2 `[Core]` — Temperature experiment

Run the same creative prompt three times:
- Once at `temperature=0` (greedy)
- Once at `temperature=0.7` (default — balanced)
- Once at `temperature=1.2` (high creativity)

Compare the outputs. Which temperature gives the most consistent answer? Which is the most surprising?

In [ ]:
# --- SOLUTION ---
creative_prompt = (
    "Complete this sentence in a creative and unexpected way: "
    "'The data scientist looked at the graph and said...'"
)

for temp in [0.0, 0.7, 1.2]:
    reply = llm.generate(creative_prompt, temperature=temp, max_new_tokens=80)
    print(f"\n─── temperature = {temp} ───")
    print(reply)

> **Key observation:** At `temperature=0` the model gives the same answer every run. At higher temperatures, outputs diverge — both more interesting and less predictable. For factual tasks (e.g., answering a customer question), keep temperature low. For creative tasks (brainstorming, writing), higher temperatures are useful.
>
> For the rest of this lab we use `temperature=0` when we want reproducible outputs to compare.

---
## 4 · The hallucination problem — live demo

This is the most important section of this lab. We are going to deliberately make the LLM hallucinate — and then understand *why* it happens, not as a bug, but as a structural property.

**Background:** TechRetail Co. is a fictional Colombian electronics retailer. Their return policy for electronics is **15 days** (not the usual 30 days). Their express shipping costs **39,900 COP**. These numbers are not in any public dataset or website — the LLM has never seen them.

### Exercise 2.1.3 `[Core]` — Trigger a hallucination

Ask the LLM the two questions below. Use `temperature=0` for reproducibility.
1. What is the return window for electronics at TechRetail Co.?
2. How much does express shipping cost at TechRetail Co.?

Observe what the model answers. Does it refuse to answer? Does it make something up? Does it acknowledge uncertainty?

In [ ]:
# --- SOLUTION ---
q1 = "What is the return window for electronics at TechRetail Co.?"
print("Q:", q1)
print("A:", llm.generate(q1, temperature=0))

In [ ]:
q2 = "How much does express shipping cost at TechRetail Co.?"
print("Q:", q2)
print("A:", llm.generate(q2, temperature=0))

**Ground truth** (from the TechRetail policy documents):
- Electronics return window: **15 days** (not 30)
- Express shipping: **39,900 COP**

Compare with what the LLM said. Common behaviours:
- It might give a plausible-sounding but wrong number (hallucination).
- It might hedge ('I don't have specific information about TechRetail') — better, but still unhelpful.
- It might extrapolate from similar companies it saw during training — risky in production.

### Why does this happen?

An LLM is trained to predict the next most likely token given everything before it. It has *no mechanism* to distinguish between:
- Things it actually knows from training data.
- Things that sound plausible based on patterns.

The model is optimised for fluency and coherence, not for truth. As Ji et al. (2023) put it: *hallucinations are not a bug that can be patched — they are a structural property of how these models work.*

**This is exactly why we need RAG**: instead of relying on the model's memory, we will retrieve the correct information and inject it into the prompt. That is what Lab 2.3 is about.

---
## 5 · `[Extension]` Context windows and token counting

### Exercise 2.1.4 `[Extension]` — How many tokens is your prompt?

LLMs process text as **tokens**, not characters or words. A token is roughly 4 characters on average, but varies ("ChatGPT" might be one token, while "supercalifragilistic" might be four).

Every model has a **context window** — the maximum number of tokens it can process at once (input + output together). If you exceed it, older parts of the conversation are silently dropped.

Use `llm.count_tokens(text)` to count the tokens in a few strings and build intuition.

In [ ]:
# --- SOLUTION ---
examples = [
    "Hello!",
    "Explain what machine learning is in one sentence.",
    "The quick brown fox jumps over the lazy dog. " * 20,  # repeated to get a longer text
]

for text in examples:
    n = llm.count_tokens(text)
    print(f"{n:>5} tokens | {len(text):>6} chars | preview: {text[:60]!r}")

### How big is our knowledge base in tokens?

In [ ]:
# --- SOLUTION ---
docs = load_company_kb()
full_kb_text = "\n\n".join(d["content"] for d in docs)
n_tokens = llm.count_tokens(full_kb_text)
print(f"Knowledge base: {len(docs)} documents, ~{n_tokens} tokens total")
print()
print("Could we just paste the whole KB into the prompt?")
print(f"  Llama 3.2 (1B) context: ~128,000 tokens → {'YES, easily fits' if n_tokens < 100_000 else 'maybe not'}")
print(f"  But in real enterprise settings, the KB can have millions of documents.")
print(f"  → RAG selects only the relevant chunks, keeping the prompt small.")

---
## Summary

| What we learned | Key takeaway |
|---|---|
| LLMs generate text by predicting the next token | They are powerful pattern matchers, not fact databases |
| Temperature controls sampling randomness | Low temp = consistent; high temp = creative |
| Hallucination is structural | The model cannot distinguish memory from plausible inference |
| Context windows are finite | Full KB injection is not always practical |

**Next → Lab 2.2:** We build a multi-turn chatbot with a system prompt, and observe exactly how far prompt engineering alone can take us before RAG becomes necessary.